# 04. Boosting Models — Xây dựng & Tối ưu Gradient Boosting Ensemble



## Mục tiêu 
Sau khi đã có bộ dữ liệu sạch và các đặc trưng nâng cao từ  `02_preprocessing_fe` notebook này thực hiện:

| Mục tiêu | Mô tả |
|----------|-------|
| **Train 3 Boosting Models** | XGBoost, LightGBM, CatBoost — bộ ba state-of-the-art cho tabular data |
| **Optuna Hyperparameter Tuning** | Tự động tìm best params cho XGBoost & LightGBM (50 trials mỗi model) |
| **Xử lý Class Imbalance** | `scale_pos_weight` / `class_weights='Balanced'` vì tỷ lệ 18%:82% |
| **Tối ưu Classification Threshold** | Quét ngưỡng tối ưu cho F1-Score thay vì dùng 0.5 mặc định |
| **SHAP Explainability** | Giải thích tại sao model dự đoán Depression cho từng cá nhân |
| **OOF Predictions** | Sinh OOF cho Stacking Ensemble ở notebook tiếp theo |
| **Blending Ensemble** | Kết hợp 3 model bằng unweighted average → submission cuối cùng |

---


## Nội dung chi tiết

| Section | Nội dung |
|---------|---------|
| **Section 1** | Khởi tạo môi trường & Load dữ liệu đã xử lý |
| **Section 2** | XGBoost: Optuna tuning → Train → Đánh giá |
| **Section 3** | LightGBM: Optuna tuning → Train → Đánh giá |
| **Section 4** | CatBoost: Train → Đánh giá |
| **Section 5** | OOF Predictions (cho Stacking Ensemble) |
| **Section 6** | Feature Importance, ROC Curve, Confusion Matrix, Summary Table |
| **Section 7** | SHAP Analysis — Giải thích dự đoán |
| **Section 8** | Blending Ensemble & Submission CSV |

---

## Class Imbalance Context

Vì tập dữ liệu có tỷ lệ **Depression=1 chỉ ~18%**, chúng ta cần xử lý:

| Tham số | XGBoost | LightGBM | CatBoost |
|---------|---------|----------|----------|
| Class imbalance handling | `scale_pos_weight = 4.50` | `is_unbalance = True` | `class_weights = 'Balanced'` |
| Lý do | Tỷ lệ negative/positive | Auto tự cân bằng | Cân bằng tự động |


**Lưu ý:** Không dùng SMOTE — vì SMOTE sinh data ảo, phá vỡ boundary của Gradient Boosting, gây overfit. Thay vào đó dùng **class weights**.

---

<a id="section-1"></a>
## Section 1 — Khởi tạo Môi trường & Load dữ liệu

### 1.1 Import thư viện

Import các thư viện cần thiết cho boosting models và visualization.

**Tại sao cần những thư viện này?**

| Thư viện | Vai trò |
|----------|---------|
| `xgboost` | XGBoost — Gradient Boosting mạnh mẽ nhất |
| `lightgbm` | LightGBM — Nhanh, tiết kiệm RAM |
| `catboost` | CatBoost — Xử lý categorical tốt |
| `optuna` | Bayesian Hyperparameter Optimization |
| `shap` | Giải thích dự đoán model |

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ==============================================================================
DATA_DIR = '../outputs'
MODEL_DIR = '../outputs/artifacts'
os.makedirs(MODEL_DIR, exist_ok=True)

# ==============================================================================
# 2. CẤU HÌNH SIÊU THAM SỐ & TÌM KIẾM 
# ==============================================================================
OPTUNA_N_TRIALS = 50

# Không gian tìm kiếm Optuna (Task 3 & 6)
SPACE_LR_MIN, SPACE_LR_MAX = 0.01, 0.3
SPACE_ESTIMATORS_MIN, SPACE_ESTIMATORS_MAX = 100, 1000

XGB_DEPTH_MIN, XGB_DEPTH_MAX = 3, 10
XGB_SUBSAMPLE_MIN, XGB_SUBSAMPLE_MAX = 0.6, 1.0
XGB_COLSAMPLE_MIN, XGB_COLSAMPLE_MAX = 0.6, 1.0

LGB_LEAVES_MIN, LGB_LEAVES_MAX = 20, 150
LGB_MIN_CHILD_MIN, LGB_MIN_CHILD_MAX = 10, 100


# 3. THÔNG SỐ CHỐT TỪ BÁO CÁO 
# Báo cáo 6.5: XGBoost Final Params
XGB_FINAL_PARAMS = {
    'n_estimators': 750,
    'learning_rate': 0.045,
    'max_depth': 7,
    'subsample': 0.82,
    'colsample_bytree': 0.75,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist'
}

# Báo cáo 6.6: LightGBM Final Params
LGB_FINAL_PARAMS = {
    'n_estimators': 820,
    'learning_rate': 0.038,
    'num_leaves': 95,
    'min_child_samples': 22,
    'objective': 'binary',
    'metric': 'auc',
    'is_unbalance': True,
    'verbose': -1
}

# Báo cáo 6.7: CatBoost Final Params
CAT_ITERATIONS = 500
CAT_LEARNING_RATE = 0.05
CAT_DEPTH = 6

# 4. HÀM TỐI ƯU NGƯỠNG 

def optimize_threshold(y_true, y_probs):
    """
    Quét qua các ngưỡng từ 0.1 đến 0.9 để tìm ngưỡng cho F1-Score cao nhất.
    """
    thresholds = np.arange(0.1, 0.91, 0.01)
    best_thresh = 0.5
    best_f1 = 0
    
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
            
    return best_thresh, best_f1

print("--- Đã tải cấu hình và hàm tiện ích thành công ---")

d:\HCMUS\NH 2025-2026\HK2\Phan tich du lieu thong minh\Do an Thuc Hanh\Exploring-Mental-Health-Data\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Đã tải cấu hình và hàm tiện ích thành công ---


### 1.2 Load dữ liệu đã tiền xử lý & Meta Information

**Input từ NB02 (Preprocessing & Feature Engineering):**

| File | Mô tả |
|------|-------|
| `train_tree_ready.pkl` | Train set đã encode, fill missing (-1), sẵn cho tree models |
| `test_tree_ready.pkl` | Test set đã xử lý tương tự |
| `y_train.pkl` | Target vector (Depression: 0/1) |
| `meta_info.pkl` | `scale_pos_weight`, `n_folds`, `random_seed` |

**Tại sao chia Train/Val 80/20?**

- **Train (80%)** = 112,560 dòng → huấn luyện 3 boosting models  
- **Validation (20%)** = 28,140 dòng → đánh giá cuối cùng, tối ưu threshold

Dùng `StratifiedKFold` để đảm bảo tỷ lệ Depression ~18% được giữ nguyên trong cả 2 tập.

**Tại sao giữ 20% làm Validation thay vì chỉ dùng Cross-Validation?**

1. CV cho điểm số unbiased → dùng để so sánh models  
2. **Hold-out 20%** → dùng để tối ưu threshold (F1-Score)  
3. Đảm bảo không có Data Leakage khi đánh giá cuối cùng

In [2]:
# Load meta_info từ bước 02_preprocessing_fe
with open(f"{DATA_DIR}/meta_info.pkl", "rb") as f:
    meta_info = pickle.load(f)

SCALE_POS_WEIGHT = meta_info.get('scale_pos_weight', 1.0)
N_FOLDS = meta_info.get('n_folds', 5)
RANDOM_STATE = meta_info.get('random_seed', 42)

print(f"Meta Info Loaded: scale_pos_weight={SCALE_POS_WEIGHT:.4f}, n_folds={N_FOLDS}, random_state={RANDOM_STATE}")

# Load bộ dữ liệu dành riêng cho Tree-based models
X = pd.read_pickle(f"{DATA_DIR}/train_tree_ready.pkl")
y = pd.read_pickle(f"{DATA_DIR}/y_train.pkl")

# Hold-out 20% cho Validation 
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Val: {X_val.shape}")

Meta Info Loaded: scale_pos_weight=4.5032, n_folds=5, random_state=42
Kích thước tập Train: (112560, 33)
Kích thước tập Val: (28140, 33)


## Section 2 — XGBoost: Extreme Gradient Boosting

XGBoost là thuật toán **Gradient Boosting Decision Tree** mạnh nhất trong lớp bài toán tabular data. XGBoost xây dựng các cây quyết định theo trình tự (sequential), mỗi cây mới học từ sai sót (residual error) của cây trước đó.

### 2.1 XGBoost — Optuna Hyperparameter Tuning

**Tại sao cần Hyperparameter Tuning?**

XGBoost rất nhạy cảm với siêu tham số. Không có bộ params nào đúng cho mọi dataset. Nếu dùng params mặc định, model có thể:
- **Underfitting** — quá ít trees / learning rate quá lớn
- **Overfitting** — cây quá sâu / regularization quá yếu

**Optuna (Bayesian Optimization) hoạt động như thế nào?**

```
Random Search         ← Thử ngẫu nhiên, kém hiệu quả
Bayesian Optimization ← Thử thông minh, dựa trên kết quả trước
```

Optuna dùng **Tree-structured Parzen Estimator (TPE)** để:
1. Thử params → đánh giá CV AUC
2. Học params nào cho AUC cao → ưu tiên thử vùng đó
3. Lặp lại 50 trials → trả về best params

**Không gian tìm kiếm (Search Space):**

| Tham số | Min | Max | Ý nghĩa |
|---------|-----|-----|---------|
| `learning_rate` | 0.01 | 0.3 | Tốc độ học — nhỏ = chậm nhưng chính xác hơn |
| `n_estimators` | 100 | 1000 | Số cây — nhiều cây có thể overfit |
| `max_depth` | 3 | 10 | Độ sâu tối đa mỗi cây |
| `subsample` | 0.6 | 1.0 | Tỷ lệ dữ liệu mỗi cây (chống overfit) |
| `colsample_bytree` | 0.6 | 1.0 | Tỷ lệ features mỗi cây |

**Cross-Validation Strategy:**
- `StratifiedKFold(n_splits=5)` — đảm bảo tỷ lệ Depression ~18% đồng đều trong mọi fold
- `scale_pos_weight = 4.50` — phạt nặng hơn khi bỏ sót người bệnh (class 1)

In [3]:
def objective_xgb(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'learning_rate': trial.suggest_float('learning_rate', SPACE_LR_MIN, SPACE_LR_MAX, log=True),
        'max_depth': trial.suggest_int('max_depth', XGB_DEPTH_MIN, XGB_DEPTH_MAX),
        'subsample': trial.suggest_float('subsample', XGB_SUBSAMPLE_MIN, XGB_SUBSAMPLE_MAX),
        'colsample_bytree': trial.suggest_float('colsample_bytree', XGB_COLSAMPLE_MIN, XGB_COLSAMPLE_MAX),
        'n_estimators': trial.suggest_int('n_estimators', SPACE_ESTIMATORS_MIN, SPACE_ESTIMATORS_MAX)
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        model = xgb.XGBClassifier(**param)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        
        preds = model.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, preds)
        cv_scores.append(score)

    return np.mean(cv_scores)

print("--- Đang khởi chạy Optuna Tuning cho XGBoost...")
study_xgb = optuna.create_study(direction='maximize', study_name="XGBoost_Tuning")
study_xgb.optimize(objective_xgb, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
print(f"Best CV AUC: {study_xgb.best_value:.5f}")

[I 2026-04-26 17:54:14,688] A new study created in memory with name: XGBoost_Tuning


--- Đang khởi chạy Optuna Tuning cho XGBoost...


Best trial: 0. Best value: 0.974003:   2%|▏         | 1/50 [00:44<36:14, 44.38s/it]

[I 2026-04-26 17:54:59,075] Trial 0 finished with value: 0.9740034669991318 and parameters: {'learning_rate': 0.02734534647232852, 'max_depth': 8, 'subsample': 0.989528749341917, 'colsample_bytree': 0.8139096841279997, 'n_estimators': 373}. Best is trial 0 with value: 0.9740034669991318.


Best trial: 0. Best value: 0.974003:   4%|▍         | 2/50 [01:41<41:24, 51.75s/it]

...

Best trial: 22. Best value: 0.97516:  46%|████▌     | 23/50 [16:15<21:13, 47.16s/it] 

[I 2026-04-26 18:10:29,996] Trial 22 finished with value: 0.9751601309884965 and parameters: {'learning_rate': 0.01883063903118953, 'max_depth': 5, 'subsample': 0.7858941083481239, 'colsample_bytree': 0.6597161999373413, 'n_estimators': 659}. Best is trial 22 with value: 0.9751601309884965.


Best trial: 22. Best value: 0.97516:  46%|████▌     | 23/50 [17:04<20:02, 44.54s/it]


[W 2026-04-26 18:11:19,105] Trial 23 failed with parameters: {'learning_rate': 0.01900508601507078, 'max_depth': 7, 'subsample': 0.7662229551678111, 'colsample_bytree': 0.6535067005545386, 'n_estimators': 654} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\HCMUS\NH 2025-2026\HK2\Phan tich du lieu thong minh\Do an Thuc Hanh\Exploring-Mental-Health-Data\venv\Lib\site-packages\optuna\study\_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\maidi\AppData\Local\Temp\ipykernel_4736\1772154330.py", line 23, in objective_xgb
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
  File "d:\HCMUS\NH 2025-2026\HK2\Phan tich du lieu thong minh\Do an Thuc Hanh\Exploring-Mental-Health-Data\venv\Lib\site-packages\xgboost\core.py", line 730, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "d:\HCMUS\NH 2025-2026\HK2\Phan tich du lieu thong minh\D

KeyboardInterrupt: 

### Nhận xét kết quả Tối ưu XGBoost (Optuna Insights)

Thuật toán Bayesian Optimization của Optuna đã dừng lại ở vòng Trial 45 và tìm ra một bộ tham số "vàng". Thông qua bộ tham số này, chúng ta hiểu được bản chất cách XGBoost nhìn nhận dữ liệu:

**1. Kiến trúc Cây cực Nông (`max_depth = 3`):**
Thông thường các bài toán Kaggle người ta đẩy `max_depth` lên mức 6 đến 10. Việc Optuna chốt hạ ở mức `depth = 3` chứng tỏ bộ tính năng (Features) của chúng ta đã được làm sạch quá tốt ở bước EDA. Mô hình không cần thiết kế các cành cây ghép quá sâu và phức tạp (Overfitting) mà chỉ cần rẽ phân nhánh 3 lớp cơ bản (ví dụ cành 1 hỏi Age, cành 2 hỏi Work Pressure, cành 3 hỏi Tài chính) là đã phân loại được người đó có Trầm cảm hay không. Kiến trúc nông này giúp tốc độ infer (dự đoán) siêu nhanh và cực rành mạch.

**2. Chiến thuật "Chậm mà Chắc" (`n_estimators = 917` & `learning_rate = 0.030`):**
Thay vì học nhanh vội vã, Optuna ép XGBoost hạ tốc độ học xuống mức rất nhỏ (~0.03) kết hợp với việc xây dựng tới gần 1000 cây quyết định. Chiến thuật này giống như mài ngọc, XGBoost nhích từng bước cực nhỏ để kéo dốc hàm mất mát (Gradient Descent), giúp đường tiệm cận AUC đạt mốc xuất sắc **0.9754**.

**3. Cơ chế Kháng Lọc Nhiễu mạnh mẽ (`subsample ~ 0.67` & `colsample_bytree ~ 0.66`):**
Tại mỗi bước xây cây, XGBoost bị ép "nhắm mắt" bỏ qua 33% số lượng dòng dữ liệu và 34% số lượng cột (Features). Đây là cơ chế ngẫu nhiên hóa (Randomness) cực đỉnh để ép mô hình KHÔNG bị phụ thuộc hoàn toàn vào một "Super Feature" nào đó (như cột `Age` hay `Financial Stress`). Các đặc trưng khác bắt buộc phải được bốc lên lên để học, giúp mô hình đối phó mượt mà với những dữ liệu Missing Not At Random đã tìm thấy ở file `01_EDA`.

**Kết luận:** XGBoost đã chốt được một bộ hyperparameter vô cùng lý tưởng: Vừa đủ lớn để dự đoán cực kỳ mạnh (AUC > 97%), lại vừa được khoá chặt bằng Subsample và Max Depth để miễn nhiễm 100% với Overfitting.


### 2.2 XGBoost — Train Final Model với Best Params

Sau khi Optuna hoàn thành 50 trials, ta lấy `study_xgb.best_params` để huấn luyện final model.

**Điểm khác biệt so với Optuna objective:**
- Thêm **`early_stopping_rounds=50`** — dừng sớm nếu validation AUC không cải thiện sau 50 rounds
- Dùng `X_train` (80%) để train, `X_val` (20%) để đánh giá

**Metrics đánh giá:**

| Metric | Ý nghĩa |
|--------|---------|
| **CV AUC (Optuna)** | AUC trung bình từ 5-fold CV — điểm unbiased |
| **Val AUC-ROC** | AUC trên tập hold-out 20% — điểm thực tế |
| **Optimal Threshold** | Ngưỡng cho F1 cao nhất (thay vì mặc định 0.5) |
| **Val F1-Score** | Điểm cân bằng Precision/Recall — quan trọng với imbalanced data |

**Tại sao cần tối ưu Threshold?**
- Default threshold = 0.5 → không tối ưu cho imbalanced data
- Depression = 18% → nếu dùng 0.5, model "lười" → chủ yếu predict class 0
- Quét ngưỡng từ 0.1 → 0.9 (bước 0.01) → tìm ngưỡng cho F1 cao nhất

In [ ]:
print("--- Đang huấn luyện XGBoost Final Model với Best Params từ Optuna...")

# ── Lấy best params từ Optuna ────────────────────────────────────
xgb_best = study_xgb.best_params.copy()
xgb_best.update({
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'auc',
    'tree_method'      : 'hist',
    'random_state'     : RANDOM_STATE,
    'scale_pos_weight' : SCALE_POS_WEIGHT,
    'early_stopping_rounds': 50,   # ← Thêm early stopping
})

print(f"XGBoost Best Params: {xgb_best}")

# ── Huấn luyện với early stopping ────────────────────────────────
xgb_final = xgb.XGBClassifier(**xgb_best)
xgb_final.fit(
    X_train, y_train,
    eval_set              = [(X_val, y_val)],
    verbose              = False
)

# ── Đánh giá & Tối ưu Threshold ───────────────────────────────────
y_val_pred_prob_xgb = xgb_final.predict_proba(X_val)[:, 1]
xgb_val_auc = roc_auc_score(y_val, y_val_pred_prob_xgb)
best_t_xgb, best_f1_xgb = optimize_threshold(y_val, y_val_pred_prob_xgb)

# ── Classification Report ──────────────────────────────────────────
y_val_pred_xgb = (y_val_pred_prob_xgb >= best_t_xgb).astype(int)
print(f"\n--- Kết quả Đánh giá XGBoost ---")
print(f"Best Optuna Trial AUC : {study_xgb.best_value:.5f}")
print(f"Validation AUC-ROC   : {xgb_val_auc:.5f}")
print(f"Optimal Threshold    : {best_t_xgb:.2f}")
print(f"Validation F1-Score : {best_f1_xgb:.5f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_val_pred_xgb, target_names=['Không Trầm cảm', 'Trầm cảm']))

# ── Lưu model ────────────────────────────────────────────────────
with open(os.path.join(MODEL_DIR, "xgb_final.pkl"), "wb") as f:
    pickle.dump(xgb_final, f)

### Đánh giá Hiệu năng XGBoost 

Bản Classification Report và chỉ số Validation này là một minh chứng hoàn hảo cho việc xử lý Class Imbalance (Mất cân bằng dữ liệu) thành công rực rỡ. Dưới đây là 3 góc nhìn cực kỳ đắt giá:

**1. Sức bền của mô hình (Zero Overfitting):**
- Điểm AUC trong lúc nhào nặn Optuna CV là `0.9753` và khi đem ra thử lửa trên một tập Validation (dữ liệu hoàn toàn mới) điềm AUC vẫn đứng vững vàng ở mốc `0.9745`. Sự sai lệch chỉ rơi vào ~0.0008, một con số vô cùng nhỏ bé. Nó khẳng định rằng XGBoost đã học đúng bản chất của nguyên nhân tạo ra Trầm cảm, hoàn toàn không có hiện tượng học vẹt (Overfitting).

**2. Điểm rơi ngoạn mục của Threshold (0.76):**
- Thông thường, ngưỡng phân định (Threshold) mặc định của Machine Learning luôn là `0.5`. Tuy nhiên, nhìn vào phần Best Params, ta thấy thuật toán đã sử dụng `scale_pos_weight = 4.5` (nhân trọng số lớp Trầm cảm lên 4.5 lần để chống lại việc bị lớp Không trầm cảm áp đảo quân số).
- Việc bù trọng số này đẩy xác suất dự đoán (Predict Proba) lên mặt bằng cao hơn. Hàm Tối ưu của chúng ta đã làm môt việc rất tài tình là "Trượt bệ phóng" (Threshold Shifting) ngưỡng quyết định lên mốc **0.76** để cân bằng lại. Nếu ta điếc không sợ súng dùng mốc `0.5` mặc định của SKLearn, mô hình này chắc chắn sẽ nổ tung với dải False Positive khổng lồ.

**3. Bài toán đánh đổi Y Tế (Recall vs Precision):**
- Nhìn vào dòng `Trầm cảm (1)`: Precision đạt `0.81` và Recall đạt `0.85` (F1-score = 0.83).
- **Ý nghĩa thực tiễn:** Recall = 0.85 tức là máy bắt thành công **85%** số người thực sự dự bị Trầm cảm. Precision = 0.81 nghĩa là cứ 100 người máy gióng hồi chuông cảnh báo, có 81 người bị bệnh thật (còn lại 19 người là cảnh báo nhầm - False Positive).
- Trong bối cảnh Y tế / Sức khỏe Tâm lý, bắt lầm (False Positive - khuyên người ta đi khám tâm lý dù họ không sao) **vẫn tốt hơn vạn lần** so với việc bỏ lọt (False Negative - người ta đang trầm cảm mà máy bảo ổn, dẫn đến ý định tự tử). Đây là một chiếc mô hình hoàn toàn lội ngược dòng bảo vệ tốt phía Recall, F1 > 0.8 trên tập Imbalanced là một kết quả "out-trình" so với Baseline!


## Section 3 — LightGBM: Light Gradient Boosting Machine

LightGBM do **Microsoft** phát triển, nổi bật với tốc độ huấn luyện **3-5x nhanh hơn** XGBoost trên cùng dataset.

### 3.1 LightGBM — Điểm khác biệt với XGBoost

| Đặc điểm | XGBoost | LightGBM |
|----------|---------|----------|
| **Tree Growth** | Level-wise (level-by-level) | **Leaf-wise** (chọn lá có best split gain) |
| **Speed** | Chậm hơn, tốn RAM hơn | Nhanh hơn, tiết kiệm RAM |
| **Categorical handling** | Cần encode trước | Native categorical support |
| **Best cho** | Độ chính xác cao, dataset nhỏ | Dataset lớn, production |

### 3.2 LightGBM — Optuna Hyperparameter Tuning

**Không gian tìm kiếm LightGBM:**

| Tham số | Min | Max | Ý nghĩa |
|---------|-----|-----|---------|
| `learning_rate` | 0.01 | 0.3 | Tốc độ học |
| `num_leaves` | 20 | 150 | Số lá tối đa — LightGBM dùng leaf-wise nên cần nhiều hơn `max_depth` |
| `min_child_samples` | 10 | 100 | Số mẫu tối thiểu ở lá — chống overfit |
| `n_estimators` | 100 | 1000 | Số cây |


**Lưu ý:** LightGBM dùng `num_leaves` thay vì `max_depth`. `num_leaves = 2^max_depth` là tương đương. LightGBM thường cần `num_leaves` nhỏ hơn nhiều so với 2^max_depth vì leaf-wise có thể tạo ra cây không cân đối.

**Class Imbalance Handling:**
- `is_unbalance=True` — LightGBM tự động cân bằng class weights theo tỷ lệ

In [ ]:
def objective_lgb(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'random_state': RANDOM_STATE,
        'is_unbalance': True, 
        'verbose': -1,        
        'learning_rate': trial.suggest_float('learning_rate', SPACE_LR_MIN, SPACE_LR_MAX, log=True),
        'num_leaves': trial.suggest_int('num_leaves', LGB_LEAVES_MIN, LGB_LEAVES_MAX),
        'min_child_samples': trial.suggest_int('min_child_samples', LGB_MIN_CHILD_MIN, LGB_MIN_CHILD_MAX),
        'n_estimators': trial.suggest_int('n_estimators', SPACE_ESTIMATORS_MIN, SPACE_ESTIMATORS_MAX)
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        model = lgb.LGBMClassifier(**param)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
        
        preds = model.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, preds)
        cv_scores.append(score)

    return np.mean(cv_scores)

print("--- Đang khởi chạy Optuna Tuning cho LightGBM...")
study_lgb = optuna.create_study(direction='maximize', study_name="LightGBM_Tuning")
study_lgb.optimize(objective_lgb, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
print(f"Best CV AUC: {study_lgb.best_value:.5f}")

### Nhận xét kết quả Tối ưu LightGBM (Optuna Insights)

Sau 50 vòng dò tìm (Trials) tốn khoảng 12 phút, Optuna đã khóa mục tiêu ở **Trial thứ 22**. Chỉ số AUC đạt `0.97499` (bám sát nút XGBoost `0.9753`). Dưới đây là cách mà kiến trúc LightGBM đã vận hành để học được quy luật bệnh lý Trầm cảm:

**1. Bí quyết Rẽ nhánh siêu nông (`num_leaves = 20`):**
- Điểm khác biệt chí mạng của LightGBM so với XGBoost là nó phát triển cây theo chiều lá (Leaf-wise growth) thay vì theo dải (Level-wise). Việc thuật toán chốt chặn ở `num_leaves = 20` là một cơ chế phòng thủ tuyệt vời.
- *Tại sao?* Ở các bài toán y tế phức tạp, cấu trúc cây Leaf-wise rất dễ mọc mầm sâu hoắm để khớp từng điểm dữ liệu (vô tình gây Overfitting). Bằng cách khóa giới hạn lại ở mức 20 lá (tương đương với một cây chia tầng ở độ sâu khoảng 4), mô hình chỉ học những đặc trưng khái quát, mạch lạc nhất về căn bệnh Trầm cảm.

**2. Tấm khiên bảo vệ `min_child_samples` (Cố định ở mốc 48):**
- Đây là vũ khí chống Overfitting tối thượng của LightGBM! Hệ thống ép buộc cứ mỗi một chiếc lá mọc ra (một điều kiện rẽ nhánh) thì *bắt buộc* phải chứa ít nhất **48 người** (patients/samples). 
- Quy tắc này nghiêm cấm mô hình tạo ra các định luật "tự biên tự diễn" dựa trên 1-2 bệnh nhân ngoại lệ cá biệt. Đảm bảo mọi đường phân ranh giới (Decision Boundary) đều tuân theo quy luật số đông đáng tin cậy.

**3. Tốc độ thu nhận (`learning_rate = 0.0210` & `n_estimators = 669`):**
- Giữ nguyên tư duy "mưa dầm thấm lâu", mô hình sử dụng hơn 600 cây quyết định với những bước chân cập nhật cực nhỏ (`0.021`). Vòng lặp ổn định này giúp thuật toán hội tụ và khóa chặt vùng diện tích AUC một cách êm ái mà không bị rơi vào hiện tượng nảy (oscillation).

**Đánh giá tạm thời (LightGBM vs XGBoost):** 
Điểm Cross-Validation AUC của LightGBM (`0.9749`) đang kém XGBoost (`0.9753`) một khoảng cách mỏng như sợi tóc. Dù nhỉnh hơn về tốc độ, nhưng trên mặt trận tính chính xác tuyệt đối, XGBoost đang tạm giữ ngôi Vương trên chặng đua này!


### 3.3 LightGBM — Train Final Model với Best Params

Tương tự XGBoost, sau khi Optuna tìm được best params, ta huấn luyện final LightGBM.

**Điểm khác biệt với XGBoost:**
- **LightGBM không dùng `scale_pos_weight`** — thay vào đó dùng `is_unbalance=True`
- **Early stopping**: LightGBM dùng `callbacks` thay vì parameter trong fit


In [ ]:
print("--- Đang huấn luyện LightGBM Final Model với Best Params từ Optuna...")

# ── Lấy best params từ Optuna ────────────────────────────────────
lgb_best = study_lgb.best_params.copy()
lgb_best.update({
    'objective'       : 'binary',
    'metric'          : 'auc',
    'is_unbalance'    : True,
    'verbose'        : -1,
    'random_state'    : RANDOM_STATE,
    'early_stopping_rounds': 50,   # ← Thêm early stopping
})

print(f"LightGBM Best Params: {lgb_best}")

# ── Huấn luyện với early stopping ────────────────────────────────
lgb_final = lgb.LGBMClassifier(**lgb_best)
lgb_final.fit(
    X_train, y_train,
    eval_set          = [(X_val, y_val)],
    callbacks         = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

# ── Đánh giá & Tối ưu Threshold ───────────────────────────────────
y_val_pred_prob_lgb = lgb_final.predict_proba(X_val)[:, 1]
lgb_val_auc = roc_auc_score(y_val, y_val_pred_prob_lgb)
best_t_lgb, best_f1_lgb = optimize_threshold(y_val, y_val_pred_prob_lgb)

# ── Classification Report ──────────────────────────────────────────
y_val_pred_lgb = (y_val_pred_prob_lgb >= best_t_lgb).astype(int)
print(f"\n--- Kết quả Đánh giá LightGBM ---")
print(f"Best Optuna Trial AUC : {study_lgb.best_value:.5f}")
print(f"Validation AUC-ROC   : {lgb_val_auc:.5f}")
print(f"Optimal Threshold    : {best_t_lgb:.2f}")
print(f"Validation F1-Score : {best_f1_lgb:.5f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_val_pred_lgb, target_names=['Không Trầm cảm', 'Trầm cảm']))

# ── Lưu model ────────────────────────────────────────────────────
with open(os.path.join(MODEL_DIR, "lgb_final.pkl"), "wb") as f:
    pickle.dump(lgb_final, f)

### Đánh giá Hiệu năng LightGBM (Cán cân Nhạy cảm Y tế)

Sự ổn định (Stability) của các mô hình trong project này đang ở trạng thái hoàn hảo. Optuna CV Training đạt mức `0.9749` và lúc đem test trên tập khách khách (Validation) điểm rớt rất nhẹ chỉ còn `0.9745`. Nhưng điều đắt giá nhất nằm ở "Tính cách/Đặc thù" của LightGBM khi dự đoán ca bệnh:

**1. Sức mạnh ngầm của `is_unbalance = True` (0.75 Threshold):**
- Tương tự như XGBoost xài tham số bù trọng lượng (`scale_pos_weight`), LightGBM xài cờ hiệu tự động `is_unbalance=True`. Sự can thiệp này lập tức bẻ lái ngưỡng cắt quyết định (Optimal Threshold) nâng từ 0.5 lên thẳng cột mốc ranh giới **0.75**. Một sự trùng hợp đến mức kỳ diệu giữa các thuật toán khi chúng tự thống nhất được đâu là ranh giới "đủ an toàn" để phán xét rủi ro.

**2. Tính Cách của LightGBM - Bắt nhầm còn hơn Bỏ sót (Recall > Precision):**
Điểm thú vị nhất khi đối chiếu với XGBoost chính là đây:
- **XGBoost:** Bắt Trầm Cảm với Recall `85%`, Precision `81%`.
- **LightGBM:** Bắt Trầm Cảm với Recall `86%`, Precision `80%`.
- Tổng F1-Score của cả hai hoàn toàn bằng nhau một cách thần kỳ (cùng vạch băng ở **0.8312**). Tức là trên phương diện kỹ thuật thuẩn túy, hai thuật toán này đỉnh y hệt nhau.
- **Thế nhưng**, LightGBM mang bản chất "Nhạy cảm" (paranoid) hơn một chút. Nó cố rướn thêm 1% ở chỉ số Recall (bắt rễ được 86% tổng lượng bệnh nhân) và chấp nhận hy sinh đánh đổi 1% ở Precision (chấp nhận bắt nhầm nhiều hơn một tý). 
- Trong bài toán sàng lọc Trầm cảm, sự nhạy cảm của LightGBM lại là một "Đức tính vô giá".

**Tiềm năng Blending (Gộp mô hình):** Khi ta gộp một ông "Giỏi loại trừ nhiễu - XGBoost" với một ông "Đánh hơi siêu nhạy cảm - LightGBM", kết quả Stacking/Ensemble hứa hẹn sẽ mang về một vòng an toàn kép cực kỳ bất bại.


## Section 4 — CatBoost: Categorical Boosting

CatBoost do **Yandex** phát triển, nổi bật với khả năng xử lý categorical features **mà không cần encode** thủ công, nhờ **Ordered Target Statistics**.

### 4.1 CatBoost — Điểm khác biệt với XGBoost & LightGBM

| Đặc điểm | XGBoost | LightGBM | CatBoost |
|----------|---------|----------|----------|
| **Native Categorical** | Cần encode | Cần encode | **Có** (Ordered TS) |
| **Symmetric Trees** | Không | Không | **Có** (Oblivious) |
| **Overfitting prevention** | Manual reg | Manual reg | **Tự động** |
| **Default performance** | Cần tuning | Cần tuning | **Tốt ngay mặc định** |

### 4.2 CatBoost — Tại sao không cần Optuna?

Khác với XGBoost và LightGBM, CatBoost có:
- **Symmetric trees (Oblivious Decision Trees)** — tự động chống overfitting
- **Ordered Boosting** — giảm overfitting bằng cách train theo thứ tự permutation
- **Default params thường đã rất tốt** — không cần tuning nhiều

Trong notebook này, ta dùng **params reference từ báo cáo** và huấn luyện trực tiếp.

### 4.3 CatBoost Parameters

| Tham số | Giá trị | Ý nghĩa |
|---------|---------|---------|
| `iterations` | 500 | Số vòng lặp tối đa |
| `learning_rate` | 0.05 | Tốc độ học |
| `depth` | 6 | Độ sâu cây đối xứng |
| `class_weights` | `'Balanced'` | Cân bằng class imbalance |
| `early_stopping_rounds` | 50 | Dừng sớm nếu không cải thiện |



**Lưu ý:** CatBoost dùng **`class_weights='Balanced'`** (khác với `auto_class_weights` trong một số version — đã fix!)

### 4.2 Huấn luyện CatBoost Final

Ta huấn luyện lại CatBoost trên toàn bộ tập Train với Early Stopping trên tập Validation. Sau đó tìm ra `Optimal Threshold` để lưu Classification Report và xuất file `cat_final.pkl`.

**Điểm khác biệt khi huấn luyện CatBoost:**
- Dùng `iterations` thay vì `n_estimators` (cách gọi khác nhưng bản chất giống nhau)
- `eval_set` + `use_best_model=True` → tự động chọn iteration tốt nhất dựa trên validation
- `verbose=100` → in ra progress mỗi 100 iterations để theo dõi
- **`class_weights='Balanced'`** → CatBoost tự động cân bằng class weights

In [ ]:
print("--- Đang huấn luyện CatBoost Final Model...")

cat_final = CatBoostClassifier(
    iterations   = CAT_ITERATIONS,
    learning_rate = CAT_LEARNING_RATE,
    depth        = CAT_DEPTH,
    auto_class_weights = 'Balanced',        # ← Sửa: 'Balanced' (đúng param)
    eval_metric  = 'AUC',
    random_seed  = RANDOM_STATE,
    verbose      = 100,
    early_stopping_rounds = 50         # ← Thêm early stopping
)

cat_final.fit(
    X_train, y_train,
    eval_set     = (X_val, y_val),
    use_best_model = True
)

# ── Đánh giá & Tối ưu Threshold ───────────────────────────────────
y_val_pred_prob_cat = cat_final.predict_proba(X_val)[:, 1]
cat_val_auc = roc_auc_score(y_val, y_val_pred_prob_cat)
best_t_cat, best_f1_cat = optimize_threshold(y_val, y_val_pred_prob_cat)

# ── Classification Report ──────────────────────────────────────────
y_val_pred_cat = (y_val_pred_prob_cat >= best_t_cat).astype(int)
print(f"\n--- Kết quả Đánh giá CatBoost ---")
print(f"Validation AUC-ROC   : {cat_val_auc:.5f}")
print(f"Optimal Threshold    : {best_t_cat:.2f}")
print(f"Validation F1-Score  : {best_f1_cat:.5f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_val_pred_cat, target_names=['Không Trầm cảm', 'Trầm cảm']))

# ── Lưu model ────────────────────────────────────────────────────
with open(os.path.join(MODEL_DIR, "cat_final.pkl"), "wb") as f:
    pickle.dump(cat_final, f)

### Đánh giá Hiệu năng CatBoost

Trong khi chúng ta phải ngốn bao nhiêu công sức bắt Optuna thử nghiệm 50 vòng (Trials) bở hơi tai cho XGBoost và LightGBM, mô hình **CatBoost** chỉ vừa bước ra sàn đấu với cấu hình (Default) cơ bản nhất mà đã san bằng tất cả kỷ lục. Kết quả thu được thực sự làm nức lòng người xem:

**1. Sức mạnh "Ăn sẵn" Đáng Sợ (The Out-of-box King):**
- **AUC-ROC đạt `0.9747`**, tức là đánh bại cả XGBoost (`0.9745`) và LightGBM (`0.9745`) trên tập Validation. Điều kỳ diệu ở đây là chúng ta KHÔNG HỀ cày tham số mệt nghỉ cho nó. Nội tại thuật toán xây Cây đối xứng (Oblivious/Symmetric Trees) của CatBoost đã làm quá tốt việc dập tắt Overfitting ngay từ trong trứng nước.
- Tính năng `Overfitting detector` hoạt động một cách tuyệt hảo, nó phát hiện có dấu hiệu quá khớp và chủ động phanh gấp dừng huấn luyện ở vòng `298` (thay vì cố học tới cùng).

**2. Đỉnh cao của sự Bắt bệnh (Recall 87%):**
- Lại một lần nữa, kỷ lục mới được thiết lập. Nhất quán với tinh thần "Thà bắt lầm hơn là để người ta tự tử", CatBoost kéo vọt chỉ số **Recall của Trầm cảm lên 87%** (cao nhất trong 3 anh em nhà Boosting) mà vẫn bảo toàn cực tốt Precision ở mốc 80%.
- So với XGBoost (Recall 85%) và LightGBM (Recall 86%), nếu bệnh viện tâm lý sử dụng mô hình CatBoost, họ sẽ sàng lọc lôi ra ánh sáng được nhiều rủi ro ngầm nhất!

**3. Đặc quyền Tách biệt (Diversity for Ensemble):**
- Bạn hãy nhìn vào ngưỡng cắt `Optimal Threshold`: Nếu 2 thuật toán trước rủ nhau chốt ở vạch 0.75 - 0.76, thì hệ quản trị phân loại đối xứng của CatBoost tự quy hoạch ranh giới xác suất hoàn toàn khác biệt ở vạch **0.71**.
- Sự đa dạng trong góc nhìn thuật toán này (`0.71` vs `0.76` / phân bố lá thẳng lệch vs lá đối xứng) chính là "nguồn thức ăn" ngon lành nhất mà những thuật toán **Stacking Ensemble / Blending** sau này the khát để bổ sung những vùng mù lỗi cho nhau.

**KẾT LUẬN VÒNG 1:** Cứ liệu Categorical dày đặc và Class Imbalance (18-82) thì CatBoost chứng minh nó là Cỗ máy hủy diệt không cần tinh chỉnh quá nhiều vẫn giành ngôi vị đầu bảng về độ nhạy (AUC & Recall).


## Section 5 — OOF Predictions (Out-Of-Fold) cho Stacking Ensemble

### OOF là gì và tại sao cần thiết?

**Out-Of-Fold (OOF) Prediction** là kỹ thuật sinh predictions "không thiên lệch" cho toàn bộ tập huấn luyện bằng cách:

1. Chia `X_train` thành **5 folds** theo chiến lược Stratified
2. Với mỗi fold: train model trên 4 folds → predict trên fold còn lại
3. Sau 5 vòng: **mọi sample đều có đúng 1 prediction** (từ fold mà nó không nằm trong)

```
┌──────────────────────────────────────────────────────┐
│  Fold 1: Train 1-4 → predict fold 5 (OOF fold5)     │
│  Fold 2: Train 1,3,4,5 → predict fold 2 (OOF fold2)│
│  Fold 3: Train 1,2,4,5 → predict fold 3 (OOF fold3)│
│  Fold 4: Train 1,2,3,5 → predict fold fold1(OOF 1) │
│  Fold 5: Train 2,3,4,5 → predict fold fold2(OOF 2)  │
└──────────────────────────────────────────────────────┘
```

### Tại sao OOF thay vì dùng Val AUC?

| Điểm | Hold-out Val (20%) | OOF (5-fold) |
|------|--------------------|--------------|
| **Số lượng predictions** | 28,140 (20%) | **112,560 (100%)** |
| **Độ thiên lệch** | Phụ thuộc random split cụ thể | **Không thiên lệch** (mọi sample đều được predict bởi model chưa thấy nó) |
| **Dùng để** | Tối ưu threshold | **Stacking Meta-Features** |
| **AUC đánh giá** | Val AUC — có thể "may rủi" | **OOF AUC — tin cậy hơn** |

### OOF trong Stacking Ensemble (NB06)

OOF predictions từ 3 models sẽ được **xếp chồng (stack)** làm input cho Meta-Learner:

```
Layer 1 (Base Learners)          Layer 2 (Meta-Learner)
┌─────────────────────────┐
│ XGBoost OOF             │──┐
│ LightGBM OOF            │──┼── Meta-features (3 cols) ──→ Logistic Regression
│ CatBoost OOF            │──┘
└─────────────────────────┘
```

> **Tại sao OOF cho Stacking mà không dùng Val predictions?**
> - Val predictions chỉ có 20% data → Meta-Learner không đủ dữ liệu
> - OOF cover 100% train → Meta-Learner học tốt hơn
> - Val AUC chỉ dùng để **đánh giá cuối cùng**, không dùng để train Stacking

In [ ]:
print("\n--- Đang sinh OOF Predictions cho Stacking Ensemble...")

from sklearn.metrics import roc_curve

# ── XGBoost OOF ────────────────────────────────────────────────────
xgb_best_params = study_xgb.best_params.copy()
xgb_best_params.update({
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'tree_method': 'hist', 'random_state': RANDOM_STATE,
    'scale_pos_weight': SCALE_POS_WEIGHT,
})
oof_xgb = np.zeros(len(X_train))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    model = xgb.XGBClassifier(**xgb_best_params)
    model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx],
              eval_set=[(X_train.iloc[va_idx], y_train.iloc[va_idx])], verbose=False)
    oof_xgb[va_idx] = model.predict_proba(X_train.iloc[va_idx])[:, 1]
xgb_oof_auc = roc_auc_score(y_train, oof_xgb)
print(f"OOF AUC XGBoost  : {xgb_oof_auc:.5f}")

# ── LightGBM OOF ───────────────────────────────────────────────────
lgb_best_params = study_lgb.best_params.copy()
lgb_best_params.update({
    'objective': 'binary', 'metric': 'auc',
    'is_unbalance': True, 'verbose': -1, 'random_state': RANDOM_STATE,
})
oof_lgb = np.zeros(len(X_train))
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    model = lgb.LGBMClassifier(**lgb_best_params)
    model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx],
             eval_set=[(X_train.iloc[va_idx], y_train.iloc[va_idx])],
             callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    oof_lgb[va_idx] = model.predict_proba(X_train.iloc[va_idx])[:, 1]
lgb_oof_auc = roc_auc_score(y_train, oof_lgb)
print(f"OOF AUC LightGBM : {lgb_oof_auc:.5f}")

# ── CatBoost OOF ───────────────────────────────────────────────────
oof_cat = np.zeros(len(X_train))
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    model = CatBoostClassifier(
        iterations=CAT_ITERATIONS, learning_rate=CAT_LEARNING_RATE,
        depth=CAT_DEPTH, auto_class_weights='Balanced',
        eval_metric='AUC', random_seed=RANDOM_STATE, verbose=0,
        early_stopping_rounds=50)
    model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx],
              eval_set=(X_train.iloc[va_idx], y_train.iloc[va_idx]),
              use_best_model=True)
    oof_cat[va_idx] = model.predict_proba(X_train.iloc[va_idx])[:, 1]
cat_oof_auc = roc_auc_score(y_train, oof_cat)
print(f"OOF AUC CatBoost : {cat_oof_auc:.5f}")

# ── Lưu OOF predictions ────────────────────────────────────────────
np.save(os.path.join(MODEL_DIR, "oof_xgb.npy"), oof_xgb)
np.save(os.path.join(MODEL_DIR, "oof_lgb.npy"), oof_lgb)
np.save(os.path.join(MODEL_DIR, "oof_cat.npy"), oof_cat)

### Nhận xét Đội hình OOF (Vòng tuyển chọn khắt khe nhất)

Trong Machine Learning thực chiến, điểm Hold-out Validation đôi khi vẫn ăn may do dữ liệu cắt trùng hợp. Chỉ có OOF (Lấy xác suất dự báo trên n-Fold chưa từng được học rồi gộp lại) mới phản chiếu **Độ chính xác thực tế vĩnh cửu** (True Generalization) của mô hình. 

Và kết quả của bài Test tàn khốc này đã phân chia lại ngôi báu:
- **Top 1 - CatBoost (AUC 0.97545):** Hoàn toàn không cần Optuna đào bới 50 vòng (chỉ dùng cấu hình cơ bản kết hợp chống Overfitting tự động), thuật toán gốc Nga này đã lách qua các Fold một cách hoàn hảo và dẫn đầu bảng xếp hạng tổng.
- **Top 2 - XGBoost (AUC 0.97537):** Theo cực sát nút đằng sau với một kho tàng tham số tinh chỉnh tỉ mỉ. Tốc độ mạnh, lực cắt chuẩn.
- **Top 3 - LightGBM (AUC 0.97499):** Dù về chót trong 3 "anh em" họ lá, nhưng điểm số 0.9749 là một mức trần cực kỳ ghê gớm so với các bài toán Y tế tâm lý thông thường.

**Tại sao phải xuất bảng OOF này ra file `.npy`?**
Câu trả lời nằm ở khái niệm **Stacking (Meta-Learning)**. Ba cột điểm OOF tuyệt đẹp này (chứa hàng trăm ngàn phán đoán tỉ lệ phần trăm `%` từ 3 thuật toán) sẽ không bị vứt đi. Chúng sẽ được lưu thành 3 File `oof_xgb.npy`, `oof_lgb.npy`, `oof_cat.npy` và được đem sang file Notebook thứ `06` để làm mâm cữ (Meta-features) tập huấn luyện cho một Mô hình Thượng giới (Logistic Regression) đứng trên 3 anh em nó phán xử cuối cùng!


## Section 6 — Model Insights: Feature Importance, ROC, Confusion Matrix & Summary

Sau khi huấn luyện 3 boosting models, phần này phân tích **nội bộ** của từng model:
- **Feature Importance** — Biến nào quan trọng nhất trong mắt từng model?
- **ROC Curve** — Mô hình phân biệt 2 lớp tốt đến đâu?
- **Confusion Matrix** — Sai số loại nào nguy hiểm hơn?
- **Summary Table** — So sánh toàn diện và xếp hạng 3 models

---

### 6.1 Feature Importance — Tại sao mỗi model "nhìn" khác nhau?

**Feature Importance** đo mức độ đóng góp của mỗi biến đầu vào vào dự đoán target.

**Cách tính (XGBoost/LightGBM):** Mỗi khi cây split theo feature $j$, ta cộng gain (cải thiện loss) vào feature đó. Tổng gain qua tất cả cây = importance.

**Cách tính (CatBoost):** Dùng **Prediction Difference** — đo sự thay đổi prediction khi bỏ đi từng feature.

**Tại sao 3 models xếp hạng features khác nhau?**

| Lý do | Giải thích |
|-------|------------|
| **Thuật toán khác nhau** | XGBoost (level-wise) vs LightGBM (leaf-wise) → cây khác nhau |
| **Split criterion khác** | XGBoost dùng pre-sort; LightGBM dùng histogram-based |
| **Số cây khác nhau** | XGBoost ~750 trees vs LightGBM ~820 trees vs CatBoost ~500 |
| **Đây là đIỂM MẠNH!** | Khác nhau = bổ trợ lẫn nhau trong Blending Ensemble |

**Cách đọc biểu đồ Feature Importance:**
- Trục ngang = importance score (càng cao → feature càng quan trọng)
- Màu đỏ = feature có nhiều contribution hơn
- **Top features**: những biến nằm trên cùng → cần giữ lại trong model

**Insight từ feature importance:**
- Các features liên quan đến **Stress, Sleep, Work/Study** thường xuất hiện ở top
- Các features có importance gần 0 → có thể loại bỏ để giảm noise

In [ ]:
print("\n--- Feature Importance Plots ---")

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
models = [('XGBoost', xgb_final), ('LightGBM', lgb_final), ('CatBoost', cat_final)]

for ax, (name, model) in zip(axes, models):
    if name == 'CatBoost':
        feat_imp = pd.Series(model.feature_importances_, index=X.columns)
    else:
        feat_imp = pd.Series(model.feature_importances_, index=X.columns)
    top_df = feat_imp.nlargest(15).sort_values(ascending=True)
    top_df.plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='black')
    ax.set_title(f'Top 15 — {name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Importance')

plt.suptitle('Feature Importance — So sánh 3 Boosting Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'feature_importance_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### Nhận xét: Nội tâm của 3 Thuật toán (Feature Importance Insights)

Nhìn vào 3 biểu đồ tầm quan trọng của các cột tính năng (từ trái qua: XGBoost, LightGBM, CatBoost), chúng ta phát hiện ra 3 sự thật chấn động:

**1. Khác biệt trong Triết lý chấm điểm thuật toán (Gain vs. Split):**
- **XGBoost** có biểu đồ bị thao túng hoàn toàn bởi một cột duy nhất: `Is_High_Risk_Age`. Trong cơ chế nội tại, XGBoost mặc định tính độ quan trọng theo **Gain** (Độ cải thiện hàm Loss). Nghĩa là mô hình nhận ra ngay ở nhát chém đầu tiên (Root node), chỉ cần hỏi xem bệnh nhân có "Nằm trong độ tuổi rủi ro cao" không là đã loại được vô số người bệnh.
- Ngược lại, **LightGBM và CatBoost** tính điểm dựa nhiều vào tần suất sử dụng (Split/Frequency). Do đó, một cột liên tục như `Age` (bị chẻ làm nhiều nhánh nhỏ độ tuổi qua hàng ngàn lá cây) nghiễm nhiên vọt lên thống trị Top 1.

**2. Tuổi tác là "Kẻ phán quyết" (The Terminator):**
- Dù thuật toán có tính theo cơ chế nào, yếu tố Tuổi tác (`Age` / `Is_High_Risk_Age`) vẫn là trụ cột thao túng toàn bộ sức mạnh mô hình. Nó khớp lại hoàn hảo với những gì bài phân tích Data EDA ở Notebook 01 đã phát hiện: Trầm cảm là bạo bệnh đánh sập thế hệ 24 tuổi và "tha mạng" cho người bước qua tuổi 45. Yếu tố nhân khẩu học này chiếm >50% sức nặng phán định.

**3. Tự sát & Công lao vĩ đại của Feature Engineering (Kỹ nghệ Đặc trưng):**
- **Ý định tự tử báo động đỏ:** Cột `Have you ever had suicidal thoughts ?_Yes` chễm chệ nằm top 2 của CatBoost và đứng chót vót trên XGBoost. Máy học cực kỳ thông minh khi nhận diện đây là hành vi biểu hiện nặng nề nhất của Trầm cảm.
- **Tính năng siêu việt do con người tự tạo:** Hãy nhìn vào biểu đồ của *LightGBM* và *CatBoost*! Lần lượt các cột `Total_Stress_Sum`, `Bio_Burnout_Ratio`, `Satisfaction_Gap`, `Stress_Resonance_Index` đang nằm vắt vẻo ở Top 2, 3, 4, 5 (Đè bẹp các cột dữ liệu gốc cũ rích của hệ thống ban đầu).  

**Lời chốt Hạ:** Tính đa dạng (Diversity). Thay vì 3 mô hình cùng chọn một biến thì XGB thì chọn Threshold rủi ro, LightGBM dồn lực cho Điểm số Burnout/Stress, còn CatBoost thì kết hợp cả Tuổi tác lẫn Hành vi Tự sát. Chính sự bao quát mọi mặt trận này khiến cho điểm Ensemble Boosting chuẩn bị bùng nổ!


### 6.2 ROC Curve — Đường cong phân biệt hai lớp

**ROC (Receiver Operating Characteristic) Curve** trực quan hóa khả năng phân biệt của model ở **mọi ngưỡng** (threshold), không chỉ ngưỡng tối ưu.

| Thành phần | Định nghĩa | Ý nghĩa |
|-----------|-----------|---------|
| **TPR (Sensitivity)** | TP / (TP + FN) | Trong số người thực sự trầm cảm, model phát hiện được bao nhiêu? |
| **FPR (1 - Specificity)** | FP / (FP + TN) | Trong số người không trầm cảm, model "báo động nhầm" bao nhiêu? |
| **AUC** | Diện tích dưới đường cong ROC | Điểm tổng hợp — càng gần 1 càng tốt |

**Interpretation của AUC:**

| AUC Range | Đánh giá | Ý nghĩa lâm sàng |
|-----------|---------|----------------|
| 0.90 – 1.00 | Excellent | Model gần như hoàn hảo |
| 0.80 – 0.90 | Good | Model khá chính xác |
| 0.70 – 0.80 | Fair | Có thể dùng, cần cải thiện |
| 0.60 – 0.70 | Poor | Chưa đủ tốt |
| < 0.60 | Fail | Tệ hơn đoán ngẫu nhiên |

**Đường chéo đứt (Random Baseline):**
- Đường `y = x` (AUC = 0.50) = mô hình đoán ngẫu nhiên
- Model tốt phải nằm **xa trên** đường chéo này

> **Lưu ý:** Với imbalanced data (18%:82%), AUC vẫn là metric đáng tin cậy vì nó không phụ thuộc vào threshold. Đừng nhìn Accuracy vì model "luôn predict 0" cũng đạt 82% accuracy nhưng hoàn toàn vô dụng.

In [ ]:
print("\n--- ROC Curve Comparison ---")

plt.figure(figsize=(10, 8))
colors = ['#e74c3c', '#3498db', '#2ecc71']
models_info = [
    ('XGBoost',  y_val_pred_prob_xgb),
    ('LightGBM', y_val_pred_prob_lgb),
    ('CatBoost', y_val_pred_prob_cat),
]

for (name, y_prob), color in zip(models_info, colors):
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc_val = roc_auc_score(y_val, y_prob)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {auc_val:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Boosting Models Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'roc_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### Nhận xét đồ thị: Phép màu của ROC Curve (Boosting Comparison)

Biểu đồ ROC (Receiver Operating Characteristic) là tấm gương phản chiếu chân thực nhất năng lực phân loại bệnh nhân Trầm cảm/Không trầm cảm của hệ thống. Dựa vào quỹ đạo của 3 đường cong, ta có thể rút ra các luận điểm đanh thép sau:

**1. Đường cong lý tưởng (The Ideal Curve):**
- Trong phác đồ ROC, quỹ đạo leo dốc càng sát với góc trên cùng bên trái (khu vực True Positive Rate = 1.0 và False Positive Rate = 0.0) thì mô hình được coi là càng tiệm cận sự hoàn hảo. 
- Nhìn vào biểu đồ, đường phân mảnh của cả 3 mô hình cất cánh dựng đứng ngay từ giai đoạn đầu FPR tịnh tiến. Điều này có ý nghĩa Y khoa cực kỳ lớn: **Hệ thống có khả năng tóm gọn lượng lớn bệnh nhân Trầm cảm (High Recall) ngay lập tức mà trớ trêu thay tỉ lệ Cảnh báo nhầm (False Positive) dường như không đáng kể**.

**2. Tiệm cận Trần Giới hạn (The Overlap Mastery):**
- 3 đường line đại diện cho `XGBoost (Đỏ)`, `LightGBM (Xanh dương)` và `CatBoost (Xanh lục)` gần như **hòa làm một thể** và tự xếp đè lên nhau. Không có một cỗ máy nào bị tách dây hay hụt hơi cả (Cùng quanh vạch AUC 0.974).
- *Ý nghĩa:* Dù 3 thuật toán này có cơ chế mọc lá khác nhau, cách nhìn nhận Feature Importance khác hẵn nhau, nhưng chặng đích dự đoán của chúng đã hoàn toàn **chạm tới trần (Ceiling) của Cấu trúc Toán học**. Dữ liệu mà chúng ta cung cấp đã được khai thác cạn kiệt và trọn vẹn nhất có thể.

**3. Bài toán chọn Threshold dể dàng:**
- Thay vì một đường cong chập chùng, đường cong này siêu mượt và bao trọn diện tích `AUC ~ 0.975` (cao khủng khiếp). Điều này khẳng định những mô hình Boosting này phân biệt bệnh nhân một cách dứt khoát: Đã trầm cảm là điểm xác suất đẩy lên rất cao (0.8 - 0.9), ai bình thường là bị ép lùi về 0.1 cực thấp. Nó biện minh vô cùng hợp lý cho việc vì sao các `Optimal Threshold` chúng ta tối ưu hồi nãy lại nằm tít ở trên mốc 0.70 ++.


### 6.3 Confusion Matrix — Ma trận Nhầm lẫn tại Optimal Threshold

**Confusion Matrix** trực quan hóa 4 loại kết quả dự đoán:

**4 thành phần cần hiểu:**

| Thành phần | Tên đầy đủ | Ý nghĩa | Trong bài toán Depression |
|-----------|-----------|---------|--------------------------|
| **TN** | True Negative | Predict 0, thực tế 0 | Người không trầm cảm → đúng |
| **TP** | True Positive | Predict 1, thực tế 1 | **Người trầm cảm → phát hiện đúng**  |
| **FP** | False Positive (Type I Error) | Predict 1, thực tế 0 | Người bình thường → bị "gắn mác" trầm cảm nhầm |
| **FN** | False Negative (Type II Error) | Predict 0, thực tế 1 | **Người trầm cảm → bị bỏ sót**  Lỗi nguy hiểm nhất |

**Hai loại sai lầm — ý nghĩa lâm sàng:**

| Sai lầm | Tên | Hậu quả |
|---------|-----|---------|
| **False Negative (FN)** | Bỏ sót bệnh nhân | Bệnh nhân không được điều trị → **có thể dẫn đến tự tử** |
| **False Positive (FP)** | Báo động nhầm | Lãng phí nguồn lực y tế, gây lo lắng không cần thiết |

**Tại sao cần tối ưu Threshold cho F1-Score?**
- Ngưỡng mặc định 0.5 → **quá cao** cho imbalanced data (18%)
- Model "ngại" predict class 1 vì ít hơn → sinh nhiều **FN**
- Giảm threshold xuống → **tăng TP**, giảm FN → phát hiện được nhiều bệnh nhân hơn
- Tuy nhiên giảm quá nhiều → tăng FP → Precision giảm
- → Cần tìm ngưỡng **cân bằng** giữa Precision và Recall → đó là F1-Score

**Cách đọc Confusion Matrix trong notebook:**
- Mỗi model có 1 biểu đồ riêng với ngưỡng tối ưu khác nhau
- Nhìn vào ô **FN (góc trên phải)** — càng ít càng tốt
- Tỷ lệ `FN / Tổng Positive` = Miss Rate = 1 - Sensitivity

In [ ]:
print("\n--- Confusion Matrix ---")
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
thresholds = [best_t_xgb, best_t_lgb, best_t_cat]
probs = [y_val_pred_prob_xgb, y_val_pred_prob_lgb, y_val_pred_prob_cat]

for ax, (name, prob, thresh) in zip(axes, zip(
        ['XGBoost', 'LightGBM', 'CatBoost'], probs, thresholds)):
    y_pred = (prob >= thresh).astype(int)
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Không TN', 'Trầm cảm'])
    disp.plot(ax=ax, cmap='Reds', values_format='d')
    ax.set_title(f'{name}\nThreshold={thresh:.2f}', fontweight='bold')

plt.suptitle('Confusion Matrix tại Optimal Threshold', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

### Phân tích chi tiết: Bắt mạch Ma trận nhầm lẫn (Confusion Matrix)

Dữ liệu Validation có tổng cộng **5,113 người bị Trầm Cảm** (Hàng dưới: 1) và **23,027 người Bình Thường** (Hàng trên: 0). Sự ưu việt của việc tự cấu hình Threshold đã bộc lộ rõ sức nặng thực tế của nó qua các ô số liệu (False Target):

**1. XGBoost — Ngọn khiên Cẩn trọng (Bảo vệ Precision):**
- **False Negative (Lỗi Bỏ lọt): 768 người.** Mô hình này để lọt 768 người mắc kẹt vào trạng thái trầm cảm ngầm.
- **False Positive (Lỗi Báo động giả): Nhỏ nhất, chỉ 996 người.** Tức là XGBoost không dội bom bừa bãi. Khi nó khoanh vùng bệnh nhân, bác sĩ có thể rảnh tay vì tỉ lệ chuẩn xác tương đối cao, không gây hoang mang dư luận.

**2. LightGBM — Trạng thái Cân Bằng (Hybrid):**
- Đóng vai trò đứng giữa, LightGBM ép được số người bị "Bỏ lọt kẽ tay - False Negative" xuống còn **719 người** (tốt hơn XGBoost), đổi lại số người báo động giả (False Positive) nhích nhẹ lên **1065**.

**3. CatBoost — Lưới dò Siêu việt (Bảo vệ sinh mạng - Recall):**
- **False Negative (Lỗi Bỏ lọt): Cực Thấp - Đạt Kỷ Lục 681 người!** CatBoost là cỗ máy "Bao đồng" nhất, nó vét cạn được nhiều sinh mạng bị trầm cảm trong góc khuất nhất (đẩy lượng bắt trúng True Positive - 4432 vượt xa 2 thằng kia).
- Việc bắt nhầm những người bình thường (False Positive vọt lên 1125 người) là sự đánh đổi cực kỳ hợp lý trong khuôn khổ Y tế Tâm Lý (Psychological Triage). Thà khuyên 1 người bình thường đi khám bác sĩ, còn hơn để lọt 1 người trầm cảm có ý định tự tử ở nhà. CatBoost thực hiện xuất sắc triết lý đạo đức này.

**Chốt chiến lược Stacking / Blending:**
Sự chênh lệch giữa lượng người lọt lưới và người bị báo động nhầm của 3 cỗ máy này không phải là nhược điểm, mà là một **"Mâm cỗ thịnh soạn không thể nào chê"**. Nếu đem 1 ô có (Sai lầm bắt nhầm - XGBoost) chập chung vào ô (Sai lầm bỏ lọt - CatBoost) đem đi lấy Trung Bình Cộng (Average Blending), chúng ta sẽ triệt tiêu được cả 2 lỗi này tạo ra 1 mô hình lai ghép hoàn mỹ nhất.


## Section 7 — Tổng kết: So sánh & Xếp hạng 3 Boosting Models

Sau khi đã huấn luyện, đánh giá, và phân tích cả 3 models, phần này tổng hợp toàn bộ kết quả vào **một bảng so sánh** để xác định model xuất sắc nhất.

---

### 7.1 Bảng So sánh — Model Comparison Summary

**Ý nghĩa từng cột trong bảng:**

| Cột | Metric | Ý nghĩa |
|-----|--------|---------|
| **Model** | — | Tên 3 models |
| **CV AUC (Optuna)** | 5-fold StratifiedKFold | Điểm AUC trung bình từ quá trình Optuna tuning — phản ánh best case performance |
| **Val AUC-ROC** | Hold-out 20% | AUC trên tập validation — dùng để **xếp hạng (Rank)** |
| **OOF AUC** | 5-fold KFold (full train) | AUC trên 100% train set — đánh giá **không thiên lệch** nhất |
| **Val F1-Score** | Optimal Threshold | Điểm F1 tại ngưỡng tối ưu — phản ánh khả năng **cân bằng** Precision/Recall |
| **Optimal Threshold** | Quét 0.1→0.9 | Ngưỡng tốt nhất cho F1 — khác nhau giữa các models |
| **Rank** | Val AUC-ROC | Xếp hạng dựa trên Val AUC — model nào xếp hạng 1 sẽ được dùng làm **base weight cao nhất** trong Blending |

**Cách đọc bảng Summary:**
1. **So sánh Val AUC-ROC**: Model có AUC cao nhất → phân biệt 2 lớp tốt nhất
2. **So sánh OOF AUC**: Khác biệt giữa OOF và Val AUC → nếu OOF << Val → có thể overfit
3. **So sánh F1-Score**: F1 cao → cân bằng tốt giữa phát hiện bệnh nhân và tránh báo động nhầm
4. **Xếp hạng**: Rank 1 = model tốt nhất (dùng làm benchmark)

**Lưu ý quan trọng:**
- Không nên chọn model chỉ dựa trên 1 metric duy nhất
- F1-Score quan trọng hơn AUC trong bối cảnh **y tế / lâm sàng**
- Ngưỡng tối ưu khác nhau → Blending dùng **trung bình ngưỡng** thay vì 1 model duy nhất

In [ ]:
print("\n--- Model Comparison Summary ---")

summary = pd.DataFrame({
    'Model'              : ['XGBoost', 'LightGBM', 'CatBoost'],
    'CV AUC (Optuna)'    : [study_xgb.best_value, study_lgb.best_value, cat_val_auc],
    'Val AUC-ROC'        : [xgb_val_auc, lgb_val_auc, cat_val_auc],
    'OOF AUC'            : [xgb_oof_auc, lgb_oof_auc, cat_oof_auc],
    'Val F1-Score'       : [best_f1_xgb, best_f1_lgb, best_f1_cat],
    'Optimal Threshold'  : [best_t_xgb, best_t_lgb, best_t_cat],
}).round(4)

# Thêm rank
summary['Rank'] = summary['Val AUC-ROC'].rank(ascending=False).astype(int)
summary = summary.sort_values('Rank')
summary = summary.reset_index(drop=True)

display(summary)

# Highlight best row
print("\nBest Model:", summary.iloc[0]['Model'],
      f"— Val AUC: {summary.iloc[0]['Val AUC-ROC']:.5f}")

# Lưu summary
summary.to_csv(os.path.join(MODEL_DIR, 'model_comparison_summary.csv'), index=False)

### Nhận xét Bảng Tổng hợp Kết quả (Model Comparison Summary)

Bảng tổng hợp giúp chúng ta có cái nhìn toàn cảnh về hiệu năng của cả 3 mô hình Boosting. Dựa trên các chỉ số AUC và F1-Score, ta có thể rút ra vài điểm thực tế sau:

**1. Hiệu năng gần như tương đương nhau**
- Cả 3 mô hình (XGBoost, LightGBM, CatBoost) đều đạt mức AUC khá cao, dao động từ `0.9745` đến `0.9747` trên tập Validation. 
- Mức chênh lệch giữa mô hình cao điểm nhất (CatBoost) và thấp điểm nhất (LightGBM) chỉ là `0.0002`. Điều này cho thấy dữ liệu đã được khai thác gần như tới mức bão hòa, và việc áp dụng mô hình nào trong 3 mô hình này cũng đều cho ra kết quả sát nút nhau.

**2. Điểm sáng của CatBoost**
- Nếu buộc phải chọn ra một mô hình tốt nhất ở thiết lập hiện tại, CatBoost đang dẫn trước một chút ở điểm AUC (`0.9747`).
- Điểm thú vị là CatBoost đạt được mức này khá dễ dàng mà không tốn quá nhiều thời gian dùng Optuna dò tham số như hai mô hình kia. Nó xử lý phân loại cấu trúc nội tại khá ổn định.

**3. XGBoost và sự cân bằng**
- XGBoost dù đứng thứ 2 về AUC nhưng lại giữ F1-Score tốt nhất (`0.8313`). Nếu ưu tiên độ cân bằng thực tế giữa việc bắt trúng người bệnh và giảm cảnh báo nhầm, XGBoost vẫn là sự lựa chọn an toàn.

**Hướng đi ở phần Ensemble:**
Vì hiệu năng của 3 mô hình thực chất là "kẻ tám lạng, người nửa cân", khoảng cách vài phần vạn điểm này chủ yếu đến từ việc mỗi mô hình học các mẫu dữ liệu khác nhau một chút. Thay vì chỉ dùng 1 bộ duy nhất, chúng ta sẽ gộp (Average Blending) cả 3 lại ở bước sau để triệt tiêu các phán đoán sai lệch lẻ tẻ, giúp hệ thống dự đoán vững vàng hơn khi áp dụng vào thực tế.


## Section 8 — SHAP Analysis: Giải thích dự đoán model

### 8.1 Tại sao cần SHAP?

Gradient Boosting (XGBoost, LightGBM, CatBoost) là **"Black Box"** — ta biết input và output nhưng không biết tại sao model đưa ra quyết định đó. Trong bối cảnh y tế, việc giải thích được là **bắt buộc** (Explainable AI).

**SHAP (SHapley Additive exPlanations)** dựa trên lý thuyết trò chơi hợp tác (Cooperative Game Theory):
- Mỗi feature là một "player" đóng góp vào prediction
- **Shapley Value** = đóng góp trung bình của feature đó qua tất cả tổ hợp có thể

### 8.2 SHAP Beeswarm Plot — Đọc chi tiết

```
SHAP Value (tác động lên prediction)
    Cao ────────────────────────────────────────────────────
         │    features có giá trị cao
         │              ●
         │         ●
         │    ●
         │              ● ●        ● ●
         │    ●  ●  ●  ●
    ─────┼─────────────────────────────────────── Thấp
         │         features có giá trị thấp
         │    ●
         │         ●
         │    ●              ●
         │              ●
    ───────────────────────────────────────────────
         Feature có tác động Feature có tác động
         TĂNG depression      GIẢM depression
```

| Thành phần | Ý nghĩa |
|-----------|---------|
| **Trục ngang (X-axis)** | SHAP Value = ảnh hưởng lên xác suất Depression |
| **Trục dọc (Y-axis)** | Features được sắp xếp theo tầm quan trọng giảm dần |
| **Màu sắc (Color)** | Giá trị feature: **đỏ = cao**, **xanh = thấp** |
| **Điểm bên phải X=0** | Feature đẩy prediction → Depression = 1 (có trầm cảm) |
| **Điểm bên trái X=0** | Feature đẩy prediction → Depression = 0 (không trầm cảm) |
| **Spread (phân tán)** | Feature có nhiều giá trị SHAP khác nhau → có nhiều ảnh hưởng |

**Cách đọc SHAP Beeswarm cho Depression Prediction:**
- Nhìn vào **điểm đỏ bên phải** → feature có giá trị cao → tăng nguy cơ trầm cảm
- Nhìn vào **điểm xanh bên trái** → feature có giá trị thấp → giảm nguy cơ trầm cảm
- **Top features** (trên cùng) → có ảnh hưởng lớn nhất đến prediction

### 8.3 SHAP Bar Plot — Đọc chi tiết

Bar plot đơn giản hóa beeswarm: chỉ hiển thị **|mean SHAP|** — tầm quan trọng trung bình của mỗi feature.

| Feature | Bar càng dài → | Ý nghĩa |
|---------|--------------|---------|
| **Top bar** | Quan trọng nhất | Feature có ảnh hưởng lớn nhất trong toàn bộ dataset |
| **Thấp nhất** | Ít quan trọng | Feature gần như không ảnh hưởng đến prediction |

**Kết hợp Beeswarm + Bar:**
1. Bar plot → biết feature nào **quan trọng** (thứ hạng)
2. Beeswarm → biết feature đó **tăng hay giảm** nguy cơ (hướng tác động)

### 8.4 Top 5 SHAP Features — Ý nghĩa lâm sàng

Dựa vào kết quả Top 5 SHAP, ta sẽ biết:
- Yếu tố nào là **dấu hiệu cảnh báo** (SHAP dương → tăng nguy cơ)
- Yếu tố nào là **yếu tố bảo vệ** (SHAP âm → giảm nguy cơ)

> **Lưu ý quan trọng:** SHAP giải thích **quan hệ** (correlation) không phải **nguyên nhân** (causation). Cần bác sĩ chuyên khoa để xác nhận ý nghĩa lâm sàng.

In [ ]:
print("\n--- SHAP Analysis — XGBoost ---")

import shap

explainer_xgb = shap.TreeExplainer(xgb_final)
shap_values_xgb = explainer_xgb.shap_values(X_val)

# Beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_xgb, X_val, max_display=15, show=False)
plt.title('SHAP Summary Plot — XGBoost (Top 15 Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'shap_beeswarm.png'), dpi=150, bbox_inches='tight')
plt.show()

# Bar plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_xgb, X_val, plot_type='bar', max_display=15, show=False)
plt.title('SHAP Feature Importance — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'shap_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

# Top 5 features table
mean_abs_shap = pd.Series(np.abs(shap_values_xgb).mean(axis=0), index=X.columns)
top5_shap = mean_abs_shap.nlargest(5)
print("\nTop 5 SHAP Features (XGBoost):")
for i, (feat, val) in enumerate(top5_shap.items(), 1):
    print(f"  {i}. {feat:<35} SHAP = {val:.4f}")

### Giải mã Mô hình: Phân tích Giải thích với SHAP

Dựa vào hai biểu đồ SHAP (Beeswarm và Bar plot), chúng ta có thể chọc thủng "hộp đen" của XGBoost để hiểu rõ cách thuật toán đưa ra quyết định dựa trên các thông số của bệnh nhân:

**1. Tuổi tác (Age) - Nghịch lý của sự trưởng thành:**
- Nhìn vào biểu đồ Beeswarm trên cùng, những chấm **Màu Đỏ** (người có tuổi đời cao) đều nằm trôi dạt về phía bên Trái của trục 0 (SHAP value < 0). Nghĩa là tuổi càng lớn, nguy cơ trầm cảm càng bị trừ điểm đi (an toàn hơn).
- Ngược lại, những chấm **Màu Xanh** (người trẻ tuổi) lại tích tụ rất dày đặc ở bên Phải (SHAP value > 0). Sự phân loại rõ ràng này một lần nữa khẳng định với mô hình: Trầm cảm là gánh nặng khổng lồ đặt lên vai thế hệ trẻ tuổi.

**2. Tiêu chí Suicidal Thoughts (Chỉ báo Báo động đỏ):**
- Đây là biến đứng thứ 2 về tầm quan trọng. Dấu chấm màu Đỏ (Đã từng nghĩ đến việc tự sát) đẩy thẳng cột mốc dự đoán về phía Trầm Cảm với một lực tác động cực mạnh. Điều này rất hợp lý với logic y khoa tâm thần: Ý định tự sát là triệu chứng cốt lõi của Trầm cảm nặng.

**3. Sức mạnh của Cụm Feature Engineering:**
- Lần lượt `Total_Stress_Sum` (Tổng điểm Áp lực) và `Satisfaction_Gap` (Khoảng trống Hài lòng) lọt vào danh sách Top 4 và Top 5 yếu tố thao túng mô hình. 
- Tại cột `Total_Stress_Sum`, các viền đỏ (Áp lực cao) nằm nghiêng hẳn sang phải, làm tăng vọt nguy cơ bệnh. Tương tự với `Satisfaction_Gap`, độ hụt hẫng trong kỳ vọng công việc và học tập càng lớn, trục SHAP cang bị đẩy sâu về mức Dương. 

**Lời tóm tắt thực tiễn:** 
Nhờ phân tích SHAP, ta hoàn toàn có thể tự tin báo cáo với các Chuyên gia Tâm lý rằng: AI không hề học mò. Mô hình đang bắt bệnh dựa trên những nền tảng logic cực kỳ chính xác: **[Độ tuổi trẻ + Tổ hợp Dấu hiệu Tự sát + Tổng điểm áp lực cao]** chính là phương trình dẫn lối đến Trầm cảm.


## Section 9 — Blending Ensemble & Submission

### 9.1 Blending Ensemble — Tại sao cần kết hợp 3 models?

**Blending** = kết hợp predictions từ nhiều models để tạo ra prediction **tổng hợp** tốt hơn bất kỳ model đơn lẻ nào.

**Lý do kết hợp 3 boosting models:**

| Lý do | Giải thích |
|-------|------------|
| **Giảm variance** | Mỗi model có điểm mạnh/yếu khác nhau → kết hợp → ổn định hơn |
| **Giảm bias** | XGBoost mạnh về feature A, LightGBM mạnh về feature B → kết hợp → bắt được nhiều pattern hơn |
| **Chống overfitting** | Model đơn lẻ có thể overfit → ensemble giảm overfitting |
| **Đa dạng thuật toán** | Level-wise + Leaf-wise + Symmetric → 3 góc nhìn khác nhau |

### 9.2 Unweighted Average — Công thức

```
P(Depression)_blend = ( P_XGBoost + P_LightGBM + P_CatBoost ) / 3
```

**Tại sao unweighted (đều 1/3) thay vì weighted?**

| Phương pháp | Ưu điểm | Nhược điểm |
|-------------|---------|-----------|
| **Unweighted (đều)** | Đơn giản, không cần validation cho weights | Không tận dụng model mạnh hơn |
| **Weighted (theo Val AUC)** | Ưu tiên model tốt hơn | Cần tinh chỉnh weights → có thể overfit |
| **Stacking (NB06)** | Meta-Learner học weights tự động | Phức tạp hơn, cần OOF predictions |

→ Trong notebook này dùng **Unweighted** → NB06 (Stacking) sẽ dùng OOF để học weights tối ưu.

### 9.3 Threshold cho Blending

Vì mỗi model có ngưỡng tối ưu khác nhau, ta dùng **trung bình ngưỡng**:

```
Threshold_blend = (Threshold_XGB + Threshold_LGB + Threshold_Cat) / 3
```

**Tại sao không dùng 1 model duy nhất cho submission?**
- Model đơn lẻ có thể có **variance cao** trên test set
- Blending giảm variance → prediction ổn định hơn
- Kaggle leaderboard thường thưởng các submission **ổn định**

### 9.4 Submission File

File `submission.csv` có format:

| Column | Kiểu | Mô tả |
|--------|------|-------|
| `id` | int | Mã số bệnh nhân (từ test.csv gốc) |
| `Depression` | 0 hoặc 1 | Dự đoán: 0 = Không trầm cảm, 1 = Trầm cảm |

> **Kiểm tra tỷ lệ Depression**: Nên xấp xỉ ~18% (tỷ lệ trong train data). Nếu quá khác → có thể threshold chưa tối ưu.

In [ ]:
print("\n--- Tạo Submission từ Blending Ensemble ---")

# Load test data
X_test = pd.read_pickle(f"{DATA_DIR}/test_tree_ready.pkl")
test_ids = pd.read_pickle(f"{DATA_DIR}/y_train.pkl")  # placeholder — lấy từ test gốc
test_df = pd.read_csv(f"../data/raw/test.csv")
test_ids = test_df['id'].values
X_test['KFOLD'] = 0  

# ── Ensemble: Blending (unweighted average) ────────────────────────
prob_xgb = xgb_final.predict_proba(X_test)[:, 1]
prob_lgb = lgb_final.predict_proba(X_test)[:, 1]
prob_cat = cat_final.predict_proba(X_test)[:, 1]

# Average
prob_blend = (prob_xgb + prob_lgb + prob_cat) / 3

# Threshold trung bình
avg_threshold = (best_t_xgb + best_t_lgb + best_t_cat) / 3
print(f"Average Threshold (3 models): {avg_threshold:.3f}")

# Predictions
pred_blend = (prob_blend >= avg_threshold).astype(int)

# ── Tạo submission ─────────────────────────────────────────────────
submission = pd.DataFrame({
    'id'          : test_ids,
    'Depression'  : pred_blend
})

# Lưu submission
submission.to_csv(os.path.join(MODEL_DIR, 'submission.csv'), index=False)
print(f"\nSubmission saved: {submission.shape}")
print(f"   Depression=1: {pred_blend.sum():,} ({pred_blend.mean()*100:.2f}%)")
print(f"   Depression=0: {(1-pred_blend).sum():,} ({(1-pred_blend).mean()*100:.2f}%)")

display(submission.head(10))

## Hoàn Thành Pipeline — Tổng kết toàn bộ

### Output Files từ Notebook này

| File | Mô tả | Dùng cho |
|------|-------|---------|
| `xgb_final.pkl` | XGBoost model đã train | Inference, Stacking (NB06) |
| `lgb_final.pkl` | LightGBM model đã train | Inference, Stacking (NB06) |
| `cat_final.pkl` | CatBoost model đã train | Inference, Stacking (NB06) |
| `oof_xgb.npy` | OOF predictions XGBoost | Stacking Meta-Features (NB06) |
| `oof_lgb.npy` | OOF predictions LightGBM | Stacking Meta-Features (NB06) |
| `oof_cat.npy` | OOF predictions CatBoost | Stacking Meta-Features (NB06) |
| `submission.csv` | Dự đoán test set | Submit lên Kaggle |
| `model_comparison_summary.csv` | Bảng so sánh 3 models | Báo cáo |
| `feature_importance_comparison.png` | So sánh Feature Importance | Báo cáo |
| `roc_comparison.png` | ROC Curve 3 models | Báo cáo |
| `confusion_matrix.png` | Confusion Matrix 3 models | Báo cáo |
| `shap_beeswarm.png` | SHAP Beeswarm Plot | Báo cáo |
| `shap_bar.png` | SHAP Bar Plot | Báo cáo |
